# Rotational Trading

Rotational trading involves buying top-performing assets and selling underperforming ones. **PyBroker** can be used for backtesting these strategies.

In [1]:
import pybroker
from pybroker import ExecContext, Strategy, YFinance

Our strategy will involve ranking and buying stocks with the highest [price rate-of-change (ROC)](https://www.investopedia.com/terms/p/pricerateofchange.asp). To start, we'll define a 20-day ROC indicator using [TA-Lib](https://github.com/TA-Lib/ta-lib-python):

In [2]:
import talib as ta

roc_20 = pybroker.indicator(
    "roc_20", lambda data: ta.ROC(data.adj_close, timeperiod=20)
)

Next, let's define the rules of our strategy:

- Buy the two stocks with the highest 20-day ROC.
- Allocate 50% of our capital to each stock.
- If either of the stocks is no longer ranked among the top five 20-day ROCs, then we will liquidate that stock.
- Trade these rules daily.

To implement the strategy, we write a `rotate` function that sets each stock's [long_score](https://www.pybroker.com/en/latest/reference/pybroker.context.html#pybroker.context.ExecContext.long_score) to its 20-day ROC. **PyBroker** then ranks the stocks by their `long_score` in descending order.

In [3]:
def rotate(ctx: ExecContext):
    ctx.long_score = ctx.indicator("roc_20")[-1]

Now that we have a method for scoring stocks by their ROC, we can use the [enable_rotation](https://www.pybroker.com/en/latest/reference/pybroker.strategy.html#pybroker.strategy.Strategy.enable_rotation) method to turn on rotational trading.

We cap long positions at `2` with [set_max_long_positions](https://www.pybroker.com/en/latest/reference/pybroker.strategy.html#pybroker.strategy.Strategy.set_max_long_positions). Setting `worst_rank_held` to `5` liquidates any currently held stock that falls outside the top five 20-day ROC rankings. Otherwise, **PyBroker** buys up to the top two ranked stocks, allocating 50% of the capital to each. This backtest uses a universe of 10 stocks:

In [4]:
strategy = Strategy(YFinance(), start_date="1/1/2018", end_date="1/1/2023")
strategy.set_max_long_positions(2)
strategy.enable_rotation(worst_rank_held=5)
strategy.add_execution(
    rotate,
    [
        "TSLA",
        "NFLX",
        "AAPL",
        "NVDA",
        "AMZN",
        "MSFT",
        "GOOG",
        "AMD",
        "INTC",
        "META",
    ],
    indicators=roc_20,
)
result = strategy.backtest(warmup=20)

Backtesting: 2018-01-01 00:00:00 to 2023-01-01 00:00:00



Loading bar data...


[                       0%                       ]

[**********            20%                       ]  2 of 10 completed

[**************        30%                       ]  3 of 10 completed

...

  0% (0 of 1259) |                       | Elapsed Time: 0:00:00 ETA:  --:--:--

 48% (611 of 1259) |##########           | Elapsed Time: 0:00:00 ETA:   0:00:00

 91% (1151 of 1259) |##################  | Elapsed Time: 0:00:00 ETA:   0:00:00

100% (1259 of 1259) |####################| Elapsed Time: 0:00:00 Time:  0:00:00

Finished backtest: 0:00:01


In [5]:
result.orders

,type,symbol,date,created,order_type,intent,shares,limit_price,market_price,fill_price,fees
id,,,,,,,,,,,
1,buy,NFLX,2018-02-01,2018-01-31,market,buy_to_open,1849,NaN,26.77,26.77,0.0
2,buy,AMD,2018-02-01,2018-01-31,market,buy_to_open,3639,NaN,13.53,13.53,0.0
3,sell,AMD,2018-02-05,2018-02-02,market,sell_to_close,3639,NaN,11.56,11.56,0.0
4,buy,AMZN,2018-02-05,2018-02-02,market,buy_to_open,623,NaN,69.49,69.49,0.0
5,sell,AMZN,2018-04-03,2018-04-02,market,sell_to_close,623,NaN,69.23,69.23,0.0
...,...,...,...,...,...,...,...,...,...,...,...
256,buy,AMD,2022-11-21,2022-11-18,market,buy_to_open,3591,NaN,72.28,72.28,0.0
257,sell,AMD,2022-12-14,2022-12-13,market,sell_to_close,3591,NaN,70.16,70.16,0.0
258,buy,NFLX,2022-12-14,2022-12-13,market,buy_to_open,8822,NaN,31.96,31.96,0.0


## Custom Position Sizing

**PyBroker** will allocate our capital equally between positions by default. To customize this, we can pass a ``sizer`` function to [enable_rotation](https://www.pybroker.com/en/latest/reference/pybroker.strategy.html#pybroker.strategy.Strategy.enable_rotation). The ``sizer`` is called with a [RotationContext](https://www.pybroker.com/en/latest/reference/pybroker.context.html#pybroker.context.RotationContext) after rotation has decided which stocks to buy, allowing us to override the size of each entry. The ``long_ranks`` attribute contains the rank of each stock, where ``1`` is the highest ranked.

Let's reuse our strategy, but this time allocate 70% of our capital to the top-ranked stock and 30% to the second:

In [6]:
from pybroker import RotationContext


def size_by_rank(rotation: RotationContext):
    for symbol, ctx in rotation.ctxs.items():
        if ctx.buy_shares is not None:
            rank = rotation.long_ranks[symbol]
            match rank:
                case 1:
                    ctx.buy_shares = ctx.calc_target_shares(0.7)
                case 2:
                    ctx.buy_shares = ctx.calc_target_shares(0.3)


strategy.enable_rotation(worst_rank_held=5, sizer=size_by_rank)
result = strategy.backtest(warmup=20)
result.orders

Backtesting: 2018-01-01 00:00:00 to 2023-01-01 00:00:00



Loading bar data...


[                       0%                       ]

[**************        30%                       ]  3 of 10 completed

[**********************50%                       ]  5 of 10 completed

...

 48% (611 of 1259) |##########           | Elapsed Time: 0:00:00 ETA:   0:00:00

 99% (1251 of 1259) |################### | Elapsed Time: 0:00:00 ETA:   0:00:00

100% (1259 of 1259) |####################| Elapsed Time: 0:00:00 Time:  0:00:00

Finished backtest: 0:00:00


,type,symbol,date,created,order_type,intent,shares,limit_price,market_price,fill_price,fees
id,,,,,,,,,,,
1,buy,NFLX,2018-02-01,2018-01-31,market,buy_to_open,2589,NaN,26.77,26.77,0.0
2,buy,AMD,2018-02-01,2018-01-31,market,buy_to_open,2183,NaN,13.53,13.53,0.0
3,sell,AMD,2018-02-05,2018-02-02,market,sell_to_close,2183,NaN,11.56,11.56,0.0
4,buy,AMZN,2018-02-05,2018-02-02,market,buy_to_open,379,NaN,69.49,69.49,0.0
5,sell,AMZN,2018-04-03,2018-04-02,market,sell_to_close,379,NaN,69.23,69.23,0.0
...,...,...,...,...,...,...,...,...,...,...,...
256,buy,AMD,2022-11-21,2022-11-18,market,buy_to_open,4701,NaN,72.28,72.28,0.0
257,sell,AMD,2022-12-14,2022-12-13,market,sell_to_close,4701,NaN,70.16,70.16,0.0
258,buy,NFLX,2022-12-14,2022-12-13,market,buy_to_open,4790,NaN,31.96,31.96,0.0
